In [1]:
!pip install tensorflow==2.10.0 scikit-learn pandas "numpy<2.0" matplotlib fastapi uvicorn
!pip install --upgrade "mistralai>=1.0.0" -q

# Install & Import Dulu Semua yang Dibutuhkan

In [1]:
import pandas as pd
import numpy as np
import re, ast, os, pickle, json, time, requests, pickle
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from datetime import datetime
from collections import Counter

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
import keras
from keras import layers, regularizers

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("TensorFlow:", tf.__version__)
print("GPU:", len(tf.config.list_physical_devices('GPU')) > 0)

TensorFlow: 2.10.0
GPU: True


# Load dan Pahami Dataset

In [2]:
df = pd.read_csv('resume_features.csv')

print(f"Total CV  : {df.shape[0]}")
print(f"Kolom     : {df.columns.tolist()}")
print(f"Kategori unik: {df['category'].nunique()}")
print()
print(df['category'].value_counts())

Total CV  : 12755
Kolom     : ['category', 'summary', 'highlights', 'experience', 'experience_years', 'education', 'certifications', 'skills', 'skills_list', 'skills_count']
Kategori unik: 44

category
HEALTH AND FITNESS           460
EDUCATION                    421
BUILDING AND CONSTRUCTION    416
FINANCE                      413
ACCOUNTANT                   412
SALES                        410
CONSULTANT                   406
BUSINESS ANALYST             385
HUMAN RESOURCES              379
PUBLIC RELATIONS             376
AVIATION                     372
DIGITAL MEDIA                366
BANKING                      360
APPAREL                      359
ADVOCATE                     339
INFORMATION TECHNOLOGY       338
DESIGNER                     315
ELECTRICAL ENGINEERING       312
OPERATIONS MANAGER           310
AUTOMOBILE                   299
MANAGEMENT                   288
ARTS                         285
MECHANICAL ENGINEER          279
SQL DEVELOPER                270
WEB DE

# Bersihkan dan Siapkan Fitur

In [3]:
def fill_nulls(df):
    df = df.copy()
    df['highlights']     = df['highlights'].fillna('no highlights')
    df['education']      = df['education'].fillna('no education information')
    df['certifications'] = df['certifications'].fillna('no certification')
    return df

df = fill_nulls(df)
print("Null setelah diisi:")
print(df.isnull().sum())

Null setelah diisi:
category            0
summary             0
highlights          0
experience          0
experience_years    0
education           0
certifications      0
skills              0
skills_list         0
skills_count        0
dtype: int64


In [4]:
def group_career_targets(category):
    cat = str(category).upper()

    if any(w in cat for w in ['ARCHITECTURE', 'CIVIL ENGINEER', 'BUILDING AND CONSTRUCTION']):
        return 'Civil, Architecture & Construction'
    if any(w in cat for w in ['MECHANICAL ENGINEER','ELECTRICAL ENGINEERING','ENGINEERING']):
        return 'Mechanical & Electrical Engineering'
    if any(w in cat for w in ['WEB DESIGNING','REACT DEVELOPER']):
        return 'Web & Front-End Development'
    if any(w in cat for w in ['DOTNET DEVELOPER', 'JAVA DEVELOPER']):
        return 'Back-End & Enterprise Architecture'
    if any(w in cat for w in ['DATABASE', 'SQL DEVELOPER', 'ETL DEVELOPER']):
        return 'Data Engineering & Governance'
    if any(w in cat for w in ['DATA SCIENCE', 'PYTHON DEVELOPER', 'BLOCKCHAIN']):
        return 'Data Science & Advanced Tech'
    if any(w in cat for w in ['DEVOPS', 'NETWORK SECURITY ENGINEER', 'TESTING']):
        return 'Infrastructure & Cloud DevOps'
    if any(w in cat for w in ['FINANCE', 'BANKING']):
        return 'Corporate Finance & Banking'
    if any(w in cat for w in ['ACCOUNTANT']):
        return 'Accounting & Tax'
    if any(w in cat for w in ['BUSINESS ANALYST', 'PMO', 'MANAGEMENT']):
        return 'Business, PMO & Analytics'
    if any(w in cat for w in ['OPERATIONS MANAGER', 'SAP DEVELOPER']):
        return 'Supply Chain & Operations'
    if any(w in cat for w in ['HUMAN RESOURCES', 'BPO']):
        return 'Human Resources & Corporate Ops'
    if any(w in cat for w in ['SALES', 'PUBLIC RELATIONS']):
        return 'Sales & Public Relations'
    if any(w in cat for w in ['CONSULTANT', 'ADVOCATE']):
        return 'Consultant & Legal Advocacy'
    if any(w in cat for w in ['DIGITAL MEDIA']):
        return 'Digital Media & Communications'
    if any(w in cat for w in ['DESIGNER', 'ARTS']):
        return 'Creative Arts & Design'
    if any(w in cat for w in ['APPAREL']):
        return 'Apparel'
    if any(w in cat for w in ['AVIATION', 'AUTOMOBILE']):
        return 'Aviation & Automotive Transport'
    if any(w in cat for w in ['AGRICULTURE', 'FOOD AND BEVERAGES']):
        return 'Agriculture & Food Technology'
    if any(w in cat for w in ['EDUCATION']):
        return 'Education & Training'
    if any(w in cat for w in ['INFORMATION TECHNOLOGY']):
        return 'Information Technology Support'
    if any(w in cat for w in ['HEALTH AND FITNESS']):
        return 'Health, Wellness & Fitness'

    return 'Other Professionals'

df['category_grouped'] = df['category'].apply(group_career_targets)

other = df[df['category_grouped'] == 'Other Professionals']['category'].unique()
if len(other) > 0:
    print(f"⚠️  Kategori tidak terpetakan: {other}")
else:
    print("✅ Semua kategori terpetakan!")

print("\nDistribusi 22 Kelompok Karir:")
dist = df['category_grouped'].value_counts()
print(dist)
print(f"\nTotal kelompok: {dist.shape[0]}")

✅ Semua kategori terpetakan!

Distribusi 22 Kelompok Karir:
category_grouped
Civil, Architecture & Construction     941
Business, PMO & Analytics              886
Sales & Public Relations               786
Corporate Finance & Banking            773
Consultant & Legal Advocacy            745
Infrastructure & Cloud DevOps          736
Mechanical & Electrical Engineering    695
Aviation & Automotive Transport        671
Data Engineering & Governance          657
Creative Arts & Design                 600
Human Resources & Corporate Ops        558
Supply Chain & Operations              542
Back-End & Enterprise Architecture     496
Health, Wellness & Fitness             460
Data Science & Advanced Tech           456
Agriculture & Food Technology          441
Education & Training                   421
Web & Front-End Development            416
Accounting & Tax                       412
Digital Media & Communications         366
Apparel                                359
Information Technolo

In [5]:
def engineer_features(df):
    df = df.copy()
    df['has_certification'] = df['certifications'].apply(lambda x: 0 if 'no certification' in str(x).lower() else 1)
    
    def count_certs(text):
        if 'no certification' in str(text).lower(): return 0
        try:
            parsed = ast.literal_eval(str(text))
            if isinstance(parsed, list): return len(parsed)
        except: pass
        return len([x for x in str(text).split(',') if x.strip()])

    df['cert_count']        = df['certifications'].apply(count_certs)
    df['has_education']     = df['education'].apply(lambda x: 0 if 'no education' in str(x).lower() else 1)
    df['experience_years']  = df['experience_years'].clip(upper=40.0)
    df['skills_count']      = df['skills_count'].clip(upper=36)
    df['has_highlights']    = df['highlights'].apply(lambda x: 0 if 'no highlights' in str(x).lower() else 1)
    
    return df

df = engineer_features(df)

num_cols = ['has_certification','cert_count','has_education',
            'experience_years','skills_count','has_highlights']
print("Statistik Fitur Numerik:")
print(df[num_cols].describe())

Statistik Fitur Numerik:
       has_certification    cert_count  has_education  experience_years  \
count       12755.000000  12755.000000   12755.000000      12755.000000   
mean            0.400706      0.706233       0.760643         12.669573   
std             0.490061      1.782344       0.426708          9.792521   
min             0.000000      0.000000       0.000000          0.000000   
25%             0.000000      0.000000       1.000000          5.000000   
50%             0.000000      0.000000       1.000000         10.800000   
75%             1.000000      1.000000       1.000000         18.100000   
max             1.000000     67.000000       1.000000         40.000000   

       skills_count  has_highlights  
count  12755.000000    12755.000000  
mean      13.286319        0.827440  
std       10.368664        0.377881  
min        1.000000        0.000000  
25%        6.000000        1.000000  
50%       10.000000        1.000000  
75%       18.000000        1.0000

In [6]:
def combine_text_features(row):
    parts = []

    # Skills
    if pd.notna(row['skills']) and str(row['skills']).strip():
        skills_text = str(row['skills'])
        for _ in range(4):
            parts.append(skills_text)

    # Summary
    if pd.notna(row['summary']) and str(row['summary']).strip():
        parts.append(str(row['summary']))

    # Experience
    if pd.notna(row['experience']) and str(row['experience']).strip():
        parts.append(str(row['experience'])[:800])

    # Highlights & Education
    if str(row['highlights']) != 'no highlights':
        parts.append(str(row['highlights'])[:300])
    if str(row['education']) != 'no education information':
        parts.append(str(row['education'])[:300])

    return ' '.join(parts)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s,.\'\-+#/]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df['combined_text'] = df.apply(combine_text_features, axis=1).apply(clean_text)

print(f"Total awal: {len(df)}")

# Hapus CV palsu yang mengandung "lorem ipsum"
df = df[~df['combined_text'].str.contains('lorem ipsum', case=False, na=False)]

# Hapus CV yang hasil ekstraksinya terlalu sedikit
df = df[df['combined_text'].str.len() >= 500]
print(f"Total setelah dibersihkan: {len(df)}")

avg_len = int(df['combined_text'].str.len().mean())
print(f"Rata-rata panjang teks : {avg_len} karakter")
print(f"Min / Max panjang       : {df['combined_text'].str.len().min()} / {df['combined_text'].str.len().max()}")

Total awal: 12755
Total setelah dibersihkan: 12702
Rata-rata panjang teks : 2589 karakter
Min / Max panjang       : 520 / 37344


# Encode Label dan Split Data

In [8]:
le = LabelEncoder()
df['label']  = le.fit_transform(df['category_grouped'])
num_classes  = len(le.classes_)
print(f"Jumlah kelas: {num_classes}")
print("Kelas:", le.classes_.tolist())

# Class weights untuk menangani imbalance
class_weights_array = compute_class_weight(
    'balanced',
    classes=np.arange(num_classes),
    y=df['label'].values
)
class_weight_dict = dict(enumerate(class_weights_array))

NUM_NUMERIC_FEATURES = 6

NUMERIC_COLS = ['experience_years','skills_count','has_certification',
                'cert_count','has_education','has_highlights']

X_text    = df['combined_text'].values
X_numeric = df[NUMERIC_COLS].values.astype(np.float32)
y_label   = df['label'].values

# Split
X_text_train, X_text_temp, X_num_train, X_num_temp, y_train, y_temp = train_test_split(
    X_text, X_numeric, y_label,
    test_size=0.30, random_state=42, stratify=y_label
)
X_text_val, X_text_test, X_num_val, X_num_test, y_val, y_test = train_test_split(
    X_text_temp, X_num_temp, y_temp,
    test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nTrain : {len(X_text_train):>5} sampel")
print(f"Val   : {len(X_text_val):>5} sampel")
print(f"Test  : {len(X_text_test):>5} sampel")

Jumlah kelas: 22
Kelas: ['Accounting & Tax', 'Agriculture & Food Technology', 'Apparel', 'Aviation & Automotive Transport', 'Back-End & Enterprise Architecture', 'Business, PMO & Analytics', 'Civil, Architecture & Construction', 'Consultant & Legal Advocacy', 'Corporate Finance & Banking', 'Creative Arts & Design', 'Data Engineering & Governance', 'Data Science & Advanced Tech', 'Digital Media & Communications', 'Education & Training', 'Health, Wellness & Fitness', 'Human Resources & Corporate Ops', 'Information Technology Support', 'Infrastructure & Cloud DevOps', 'Mechanical & Electrical Engineering', 'Sales & Public Relations', 'Supply Chain & Operations', 'Web & Front-End Development']

Train :  8891 sampel
Val   :  1905 sampel
Test  :  1906 sampel


# Tokenisasi Teks

In [9]:
MAX_TOKENS = 20000  
MAX_LEN    = 200     
EMBED_DIM  = 192     

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=MAX_LEN,
    ngrams=2,
    standardize='lower_and_strip_punctuation'
)
vectorizer.adapt(X_text_train)

vocab_size = len(vectorizer.get_vocabulary())
print(f"Ukuran kamus: {vocab_size:,} token")

Ukuran kamus: 20,000 token


# tf.data Pipeline

In [10]:
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_val_scaled   = scaler.transform(X_num_val)
X_num_test_scaled  = scaler.transform(X_num_test)

BATCH_SIZE = 64
AUTOTUNE   = tf.data.AUTOTUNE

@tf.function
def augment_data(inputs, labels, sample_weights=None):
    text = inputs['text_input']
    numeric = inputs['numeric_input']
    
    mask = tf.random.uniform(tf.shape(text)) > 0.1
    text = tf.where(mask, text, tf.zeros_like(text))
    
    noise = tf.random.normal(tf.shape(numeric), mean=0.0, stddev=0.05)
    numeric = numeric + noise
    
    aug_inputs = {'text_input': text, 'numeric_input': numeric}
    
    if sample_weights is not None:
        return aug_inputs, labels, sample_weights
    return aug_inputs, labels

def make_dataset(X_text, X_numeric, y_labels, is_train=False):
    X_tokens = vectorizer(X_text)
    if is_train:
        sample_weights = np.array(
            [class_weight_dict[lbl] for lbl in y_labels],
            dtype=np.float32
        )
        ds = tf.data.Dataset.from_tensor_slices((
            {'text_input': X_tokens, 'numeric_input': X_numeric},
            y_labels,
            sample_weights
        )).shuffle(buffer_size=5000, seed=42)
        ds = ds.map(augment_data, num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((
            {'text_input': X_tokens, 'numeric_input': X_numeric},
            y_labels
        ))
    return ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

train_ds = make_dataset(X_text_train, X_num_train_scaled, y_train, is_train=True)
val_ds   = make_dataset(X_text_val,   X_num_val_scaled,   y_val)
test_ds  = make_dataset(X_text_test,  X_num_test_scaled,  y_test)

print(f"Train batches: {len(list(train_ds))}")
print(f"Val batches  : {len(list(val_ds))}")
print(f"Test batches : {len(list(test_ds))}")

Train batches: 139
Val batches  : 30
Test batches : 30


# Komponen Kustom Tensorflow

In [11]:
# ─── Custom Layer 1: StripMask ───────────────────────────────
@tf.keras.utils.register_keras_serializable()
class StripMask(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True

    def call(self, inputs):
        return inputs

    def compute_mask(self, inputs, mask=None):
        return None

    def get_config(self):
        return super().get_config()


# ─── Custom Layer 2: AttentionPooling ────────────
@tf.keras.utils.register_keras_serializable()
class AttentionPooling(tf.keras.layers.Layer):
    def __init__(self, units=128, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True
        self.units     = units
        self.attention = tf.keras.layers.Dense(units, activation='tanh')
        self.score     = tf.keras.layers.Dense(1)

    def call(self, inputs, mask=None, training=False):
        attn_weights = self.score(self.attention(inputs))

        if mask is not None:
            mask_f = tf.cast(tf.expand_dims(mask, -1), attn_weights.dtype)
            attn_weights = attn_weights + (1.0 - mask_f) * -1e9

        attn_weights = tf.nn.softmax(attn_weights, axis=1)
        return tf.reduce_sum(inputs * attn_weights, axis=1)

    def compute_mask(self, inputs, mask=None):
        return None

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config


# ─── Custom Loss: Focal Loss + Label Smoothing ────────────────
@tf.keras.utils.register_keras_serializable()
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, smoothing=0.1, name='focal_loss', **kwargs):
        super().__init__(name=name, reduction='none', **kwargs)
        self.gamma     = gamma
        self.smoothing = smoothing

    def call(self, y_true, y_pred):
        y_pred  = tf.cast(y_pred, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        n_cls   = tf.cast(tf.shape(y_pred)[-1], tf.float32)

        y_oh    = tf.one_hot(tf.cast(y_true, tf.int32), depth=tf.shape(y_pred)[-1])
        y_oh    = y_oh * (1.0 - self.smoothing) + (self.smoothing / n_cls)

        ce      = -tf.reduce_sum(y_oh * tf.math.log(y_pred), axis=-1)
        p_cor   = tf.reduce_sum(y_oh * y_pred, axis=-1)
        return tf.pow(1.0 - p_cor, self.gamma) * ce

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'smoothing': self.smoothing})
        return config


# ─── Custom Callback: KAVA Trainer ───────────────────────────
class KAVACallback(tf.keras.callbacks.Callback):
    def __init__(self, log_dir, patience=5, model_path='kava_best_model.keras'):
        super().__init__()
        self.log_dir       = log_dir
        self.patience      = patience
        self.model_path    = model_path
        self.best_val_loss = float('inf')
        self.wait          = 0
        self.writer        = tf.summary.create_file_writer(log_dir)

    def on_epoch_end(self, epoch, logs=None):
        logs     = logs or {}
        val_loss = logs.get('val_loss', float('inf'))
        val_acc  = logs.get('val_accuracy', 0)

        with self.writer.as_default():
            for name, val in logs.items():
                tf.summary.scalar(name, data=val, step=epoch)
            self.writer.flush()

        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.wait          = 0
            self.model.save(self.model_path)
            print(f"  ✅ Epoch {epoch+1}: Tersimpan! val_loss={val_loss:.4f} (val_acc={val_acc*100:.2f}%)")
        else:
            self.wait += 1
            print(f"  ⏳ Epoch {epoch+1}: Tidak ada penurunan val_loss ({self.wait}/{self.patience}) — best={self.best_val_loss:.4f}")
            if self.wait >= self.patience:
                self.model.stop_training = True
                print(f"  🛑 Early Stopping! Best val_loss: {self.best_val_loss:.4f}")

    def on_train_end(self, logs=None):
        print(f"\n{'='*60}")
        print(f"Training Selesai! Best Val Loss: {self.best_val_loss:.4f}")
        print(f"Model disimpan di: {self.model_path}")
        self.writer.close()

# ─── Custom Callback: Overfit Detector ───────────────────────────────
class OverfitDetector(tf.keras.callbacks.Callback):
    def __init__(self, threshold=0.15):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_acc = logs.get('accuracy', 0)
        val_acc   = logs.get('val_accuracy', 0)
        
        if (train_acc - val_acc) > self.threshold:
            print(f"  ⚠️ WARNING: Model indikasi overfitting! Gap akurasi {(train_acc - val_acc)*100:.1f}% > {self.threshold*100:.0f}%")

print("✅ Semua komponen kustom siap (3 Custom Layer, 1 Custom Loss, 1 Custom Callback)")

✅ Semua komponen kustom siap (3 Custom Layer, 1 Custom Loss, 1 Custom Callback)


# Bangun Model

In [13]:
def build_kava_model(vocab_size, embed_dim, max_len, num_numeric, num_classes):
    text_input = keras.Input(shape=(max_len,), name='text_input', dtype='int32')
    emb = layers.Embedding(vocab_size, embed_dim, mask_zero=True, name='embedding')(text_input)
    emb = layers.SpatialDropout1D(0.5)(emb)

    # Jalur A: Custom Attention Pooling
    x_attn = AttentionPooling(units=64, name='attention_pool')(emb)

    # Jalur B: BiLSTM
    x_unmasked = StripMask(name='strip_mask')(emb)
    x_lstm = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
        name='bilstm_1'
    )(x_unmasked)
    x_max = layers.GlobalMaxPooling1D(name='global_max')(x_lstm)
    x_avg = layers.GlobalAveragePooling1D(name='global_avg')(x_lstm)

    x_text = layers.Concatenate(name='text_concat')([x_attn, x_max, x_avg])
    x_text = layers.BatchNormalization()(x_text)
    x_text = layers.Dropout(0.5)(x_text)

    numeric_input = keras.Input(shape=(num_numeric,), name='numeric_input', dtype='float32')
    
    exp_years_input = numeric_input[:, 0:1] 
    other_numeric_input = numeric_input[:, 1:] 
    exp_years_weighted = exp_years_input * 3.0
    processed_numeric_input = layers.Concatenate()([exp_years_weighted, other_numeric_input])

    x_num = layers.Dense(48, activation='relu', name='numeric_dense')(processed_numeric_input)
    x_num = layers.BatchNormalization()(x_num)
    x_num = layers.Dropout(0.2)(x_num)

    # Fusion Text + Numeric
    combined = layers.Concatenate(name='fusion_concat')([x_text, x_num])
    combined = layers.ActivityRegularization(l2=1e-4)(combined)

    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     name='dense_128')(combined)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     name='dense_64')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    output = layers.Dense(
        num_classes, activation='softmax',
        dtype='float32', name='output'
    )(x)

    return keras.Model(
        inputs=[text_input, numeric_input],
        outputs=output,
        name='KAVA_v5_Optimized'
    )

model = build_kava_model(
    vocab_size   = vocab_size,
    embed_dim    = EMBED_DIM,
    max_len      = MAX_LEN,
    num_numeric  = NUM_NUMERIC_FEATURES,
    num_classes  = num_classes
)

model.summary()
print(f'Total trainable params: {model.count_params():,}')

Model: "KAVA_v5_Optimized"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text_input (InputLayer)        [(None, 200)]        0           []                               
                                                                                                  
 embedding (Embedding)          (None, 200, 192)     3840000     ['text_input[0][0]']             
                                                                                                  
 spatial_dropout1d_1 (SpatialDr  (None, 200, 192)    0           ['embedding[0][0]']              
 opout1D)                                                                                         
                                                                                                  
 numeric_input (InputLayer)     [(None, 6)]          0           []               

# Training Model

In [14]:
EPOCHS = 30

total_steps   = EPOCHS * len(train_ds)
warmup_steps  = int(total_steps * 0.10)


class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, total_steps, warmup_steps):
        self.peak_lr      = peak_lr
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)

    def __call__(self, step):
        step       = tf.cast(step, tf.float32)
        warmup_lr  = self.peak_lr * (step / self.warmup_steps)
        cosine_steps = tf.maximum(step - self.warmup_steps, 0.0)
        cosine_total = self.total_steps - self.warmup_steps
        cosine_lr    = self.peak_lr * 0.5 * (1 + tf.cos(np.pi * cosine_steps / cosine_total))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'peak_lr':      self.peak_lr,
            'total_steps':  int(self.total_steps.numpy()),
            'warmup_steps': int(self.warmup_steps.numpy())
        }


lr_schedule = WarmupCosineDecay(peak_lr=7e-4, total_steps=total_steps, warmup_steps=warmup_steps)
optimizer   = tf.keras.optimizers.Adam(
    learning_rate=lr_schedule,
    clipnorm=1.0
)
loss_fn = FocalLoss(gamma=2.0, smoothing=0.1)

train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
val_acc_metric   = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
train_mae_metric = tf.keras.metrics.MeanAbsoluteError(name='train_mae')
val_mae_metric   = tf.keras.metrics.MeanAbsoluteError(name='val_mae')

print(f'EPOCHS={EPOCHS} | peak_lr=7e-4 | warmup={warmup_steps} steps')
print(f'FocalLoss smoothing=0.1 | L2=5e-4 | Dropout max=0.5')

EPOCHS=30 | peak_lr=7e-4 | warmup=417 steps
FocalLoss smoothing=0.1 | L2=5e-4 | Dropout max=0.5


In [15]:
@tf.function
def train_step(text_batch, num_batch, y_batch, sample_weights):
    with tf.GradientTape() as tape:
        preds           = model({'text_input': text_batch, 'numeric_input': num_batch},
                                training=True)
        per_sample_loss = loss_fn(y_batch, preds)
        loss_value      = tf.reduce_mean(per_sample_loss * sample_weights)

    grads = tape.gradient(loss_value, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    train_acc_metric.update_state(y_batch, preds)
    y_oh = tf.one_hot(tf.cast(y_batch, tf.int32), depth=num_classes)
    train_mae_metric.update_state(y_oh, preds)
    return loss_value


@tf.function
def val_step(text_batch, num_batch, y_batch):
    preds  = model({'text_input': text_batch, 'numeric_input': num_batch},
                   training=False)
    v_loss = tf.reduce_mean(loss_fn(y_batch, preds))
    val_acc_metric.update_state(y_batch, preds)
    y_oh = tf.one_hot(tf.cast(y_batch, tf.int32), depth=num_classes)
    val_mae_metric.update_state(y_oh, preds)
    return v_loss


print("✅ train_step & val_step (tf.GradientTape) siap")

✅ train_step & val_step (tf.GradientTape) siap


In [16]:
os.makedirs("artifacts", exist_ok=True)

LOG_DIR    = f"logs/kava/{datetime.now().strftime('%Y%m%d-%H%M%S')}"
MODEL_PATH = 'artifacts/kava_best_model.keras'
os.makedirs(LOG_DIR, exist_ok=True)

train_writer = tf.summary.create_file_writer(f"{LOG_DIR}/train")
val_writer   = tf.summary.create_file_writer(f"{LOG_DIR}/val")
print(f"TensorBoard: tensorboard --logdir=logs/kava")
print(f"Log dir    : {LOG_DIR}")
print()

TensorBoard: tensorboard --logdir=logs/kava
Log dir    : logs/kava/20260604-124011



In [17]:
kava_cb = KAVACallback(log_dir=f"{LOG_DIR}/callback", patience=5, model_path=MODEL_PATH)
kava_cb.set_model(model)

overfit_cb = OverfitDetector(threshold=0.15)

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'train_mae':  [], 'val_mae':  []
}

for epoch in range(EPOCHS):
    start = time.time()

    # ─── Training pass ───────────────────────────────────
    total_tloss, n_train = 0.0, 0
    for (x_batch, y_batch, w_batch) in train_ds:
        loss = train_step(
            x_batch['text_input'], x_batch['numeric_input'],
            y_batch, w_batch
        )
        total_tloss += loss; n_train += 1

    t_loss = total_tloss / n_train
    t_acc  = train_acc_metric.result()
    t_mae  = train_mae_metric.result()

    # ─── Validation pass ─────────────────────────────────
    total_vloss, n_val = 0.0, 0
    for (x_batch, y_batch) in val_ds:
        vloss = val_step(x_batch['text_input'], x_batch['numeric_input'], y_batch)
        total_vloss += vloss; n_val += 1

    v_loss = total_vloss / n_val
    v_acc  = val_acc_metric.result()
    v_mae  = val_mae_metric.result()

    elapsed = time.time() - start

    # TensorBoard logging
    with train_writer.as_default():
        tf.summary.scalar('loss', t_loss, step=epoch)
        tf.summary.scalar('accuracy', float(t_acc), step=epoch)
        tf.summary.scalar('mae', float(t_mae), step=epoch)
    with val_writer.as_default():
        tf.summary.scalar('loss', v_loss, step=epoch)
        tf.summary.scalar('accuracy', float(v_acc), step=epoch)
        tf.summary.scalar('mae', float(v_mae), step=epoch)

    # Simpan history
    history['train_loss'].append(float(t_loss))
    history['val_loss'].append(float(v_loss))
    history['train_acc'].append(float(t_acc))
    history['val_acc'].append(float(v_acc))
    history['train_mae'].append(float(t_mae))
    history['val_mae'].append(float(v_mae))

    # Reset metrik
    train_acc_metric.reset_state()
    val_acc_metric.reset_state()
    train_mae_metric.reset_state()
    val_mae_metric.reset_state()

    print(f"Epoch {epoch+1:>3}/{EPOCHS} | "
          f"loss={float(t_loss):.4f} acc={float(t_acc)*100:.2f}% mae={float(t_mae):.4f} | "
          f"val_loss={float(v_loss):.4f} val_acc={float(v_acc)*100:.2f}% val_mae={float(v_mae):.4f} | "
          f"{elapsed:.1f}s")

    # KAVACallback
    logs_dict = {
        'loss': float(t_loss), 'accuracy': float(t_acc), 'mae': float(t_mae),
        'val_loss': float(v_loss), 'val_accuracy': float(v_acc), 'val_mae': float(v_mae)
    }
    
    kava_cb.on_epoch_end(epoch, logs=logs_dict)
    overfit_cb.on_epoch_end(epoch, logs=logs_dict)

    if model.stop_training:
        break

kava_cb.on_train_end()

Epoch   1/30 | loss=4.0162 acc=4.70% mae=0.0868 | val_loss=2.9169 val_acc=5.04% val_mae=0.0868 | 208.8s
  ✅ Epoch 1: Tersimpan! val_loss=2.9169 (val_acc=5.04%)
Epoch   2/30 | loss=3.6957 acc=5.77% mae=0.0861 | val_loss=2.8788 val_acc=5.62% val_mae=0.0864 | 194.6s
  ✅ Epoch 2: Tersimpan! val_loss=2.8788 (val_acc=5.62%)
Epoch   3/30 | loss=3.2104 acc=8.82% mae=0.0847 | val_loss=2.6736 val_acc=9.71% val_mae=0.0855 | 201.3s
  ✅ Epoch 3: Tersimpan! val_loss=2.6736 (val_acc=9.71%)
Epoch   4/30 | loss=2.5617 acc=17.41% mae=0.0804 | val_loss=2.1229 val_acc=28.82% val_mae=0.0798 | 205.5s
  ✅ Epoch 4: Tersimpan! val_loss=2.1229 (val_acc=28.82%)
Epoch   5/30 | loss=2.0943 acc=28.60% mae=0.0749 | val_loss=1.6396 val_acc=48.92% val_mae=0.0696 | 199.2s
  ✅ Epoch 5: Tersimpan! val_loss=1.6396 (val_acc=48.92%)
Epoch   6/30 | loss=1.7560 acc=37.32% mae=0.0703 | val_loss=1.4345 val_acc=53.91% val_mae=0.0608 | 195.2s
  ✅ Epoch 6: Tersimpan! val_loss=1.4345 (val_acc=53.91%)
Epoch   7/30 | loss=1.4974 acc=

# Evaluasi

In [18]:
best_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        'StripMask':        StripMask,
        'AttentionPooling': AttentionPooling,
        'FocalLoss':        FocalLoss
    }
)

y_pred_prob = best_model.predict(
    {'text_input': vectorizer(X_text_test),
     'numeric_input': X_num_test_scaled},
    batch_size=128, verbose=1
)
y_pred = np.argmax(y_pred_prob, axis=1)

acc      = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
y_test_oh = tf.one_hot(y_test, depth=num_classes).numpy()
final_mae = float(np.mean(np.abs(y_pred_prob - y_test_oh)))

print("=" * 60)
print(f"TEST ACCURACY : {acc*100:.2f}%  (target: ≥85%)")
print(f"MACRO F1      : {macro_f1*100:.2f}%")
print(f"MAE           : {final_mae:.5f}  (target: ≤0.02)")
print("=" * 60)

if acc >= 0.85:
    print("✅ MVP TERCAPAI: Akurasi ≥ 85%")
else:
    print(f"⚠️  Akurasi {acc*100:.2f}% — belum 85%. Coba jalankan ulang atau tambah epochs.")

if final_mae <= 0.02:
    print("✅ MVP TERCAPAI: MAE ≤ 0.02")
else:
    print(f"⚠️  MAE {final_mae:.5f} — belum 0.02")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))

15/15 [==============================] - 6s 302ms/step
TEST ACCURACY : 68.26%  (target: ≥85%)
MACRO F1      : 67.61%
MAE           : 0.04417  (target: ≤0.02)
⚠️  Akurasi 68.26% — belum 85%. Coba jalankan ulang atau tambah epochs.
⚠️  MAE 0.04417 — belum 0.02

Classification Report:
                                     precision    recall  f1-score   support

                   Accounting & Tax      0.681     0.770     0.723        61
      Agriculture & Food Technology      0.635     0.723     0.676        65
                            Apparel      0.718     0.528     0.609        53
    Aviation & Automotive Transport      0.615     0.663     0.638       101
 Back-End & Enterprise Architecture      0.713     0.893     0.793        75
          Business, PMO & Analytics      0.674     0.481     0.561       133
 Civil, Architecture & Construction      0.872     0.674     0.760       141
        Consultant & Legal Advocacy      0.538     0.375     0.442       112
        Corporate Finan

# Visualisasi Learning Curve

In [19]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_range = range(1, len(history['train_acc']) + 1)

axes[0].plot(epochs_range, history['train_acc'], label='Train')
axes[0].plot(epochs_range, history['val_acc'],   label='Val')
axes[0].axhline(0.85, color='r', linestyle='--', label='Target 85%')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_range, history['train_loss'], label='Train')
axes[1].plot(epochs_range, history['val_loss'],   label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_range, history['train_mae'], label='Train')
axes[2].plot(epochs_range, history['val_mae'],   label='Val')
axes[2].axhline(0.02, color='r', linestyle='--', label='Target MAE 0.02')
axes[2].set_title('MAE'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('kava_learning_curve.png', dpi=150, bbox_inches='tight')
print("Grafik disimpan: kava_learning_curve.png")

Grafik disimpan: kava_learning_curve.png


# Pipeline Keyword Matching

In [20]:
def build_skill_gap_database(df):
    skill_db = {}
    for group in df['category_grouped'].unique():
        group_df   = df[df['category_grouped'] == group]
        all_skills = []
        for raw in group_df['skills'].dropna():
            for s in str(raw).split(','):
                s = s.strip().lower()
                if len(s) > 2:
                    all_skills.append(s)
        from collections import Counter
        freq  = Counter(all_skills)
        top20 = [skill for skill, cnt in freq.most_common(30) if cnt >= 2][:20]
        skill_db[group] = top20
    return skill_db

skill_db = build_skill_gap_database(df)
print("✅ Skill Gap Database berhasil dibangun!")


✅ Skill Gap Database berhasil dibangun!


In [21]:
domain_corpus = {}
for role, skills in skill_db.items():
    domain_corpus[role] = " ".join(skills)

tfidf_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
tfidf_vec.fit(list(domain_corpus.values()))

domain_vectors = {
    role: tfidf_vec.transform([text]) 
    for role, text in domain_corpus.items()
}

def semantic_matching_pipeline(cv_text: str) -> dict:
    cv_vector = tfidf_vec.transform([cv_text.lower()])
    scores = {}
    for role, role_vec in domain_vectors.items():
        sim = cosine_similarity(cv_vector, role_vec)[0][0]
        scores[role] = round(sim * 100, 1)
    
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

# Uji coba
sample_cv = "python machine learning tensorflow pandas sql data visualization statistics"
kw_result = semantic_matching_pipeline(clean_text(sample_cv))
print("=== SEMANTIC MATCHING TOP-3 ===")
for domain, score in list(kw_result.items())[:3]:
    print(f"  {domain:40s} {score:.1f}%")

=== SEMANTIC MATCHING TOP-3 ===
  Data Science & Advanced Tech             46.2%
  Data Engineering & Governance            10.9%
  Business, PMO & Analytics                6.6%


# Inference: Prediksi Top-3 Role

In [22]:
def predict_top3_cv(text: str, exp_years: float, certs_count: int,
                    has_edu: int, skills_raw: str = "") -> list:
    has_cert = 1 if certs_count > 0 else 0
    # Asumsikan saat testing has_highlights = 1
    has_hl = 1 
    skills_count_est = min(len(skills_raw.split(',')) if skills_raw else 15, 36)

    num_feat   = np.array([[exp_years, skills_count_est, has_cert, certs_count,
                            has_edu, has_hl]], dtype=np.float32)
    num_scaled = scaler.transform(num_feat)

    parts_dl = []
    if skills_raw.strip():
        for _ in range(4):
            parts_dl.append(skills_raw)
    parts_dl.append(text)
    
    cleaned_dl = clean_text(' '.join(parts_dl))
    text_token = vectorizer(np.array([cleaned_dl]))
    
    probs = best_model.predict(
        {'text_input': text_token, 'numeric_input': num_scaled},
        verbose=0
    )[0]

    cleaned_tfidf = clean_text(f"{skills_raw} {text}")
    kw_scores = semantic_matching_pipeline(cleaned_tfidf)

    combined_scores = {}
    for i, role in enumerate(le.classes_):
        dl_score  = float(probs[i]) * 100
        kw_score  = kw_scores.get(role, 0.0)
        combined_scores[role] = {
            'dl_confidence': round(dl_score, 2),
            'keyword_score': kw_score,
            'final_score':   round(dl_score * 0.8 + kw_score * 0.2, 2)
        }

    top3 = sorted(combined_scores.items(),
                  key=lambda x: x[1]['final_score'], reverse=True)[:3]
    return [{'rank': i+1, 'role': role, **scores}
            for i, (role, scores) in enumerate(top3)]


# Uji coba
sample_text = """Experienced data scientist with Python, TensorFlow, scikit-learn,
pandas, numpy. Built ML models and recommendation systems. AWS and Docker."""

top3 = predict_top3_cv(
    text        = sample_text,
    exp_years   = 3.0,
    certs_count = 2,
    has_edu     = 1,
    skills_raw  = "python, tensorflow, pandas, sql, aws"
)

print("=== TOP-3 ROLE MATCHES ===")
for r in top3:
    print(f"  #{r['rank']} {r['role']:35s} DL:{r['dl_confidence']:.1f}% "
          f"SEMANTIC:{r['keyword_score']:.1f}% Final:{r['final_score']:.1f}%")

=== TOP-3 ROLE MATCHES ===
  #1 Data Science & Advanced Tech        DL:97.8% SEMANTIC:26.8% Final:83.6%
  #2 Infrastructure & Cloud DevOps       DL:0.8% SEMANTIC:15.7% Final:3.8%
  #3 Supply Chain & Operations           DL:0.0% SEMANTIC:13.5% Final:2.7%


In [23]:
# Regex & konstanta

_MONTHS = (
    r"(?:january|february|march|april|may|june|july|august|september|"
    r"october|november|december|jan|feb|mar|apr|jun|jul|aug|sep|oct|nov|dec)"
)

_DATE_RANGE_REGEX = re.compile(
    rf"({_MONTHS}\s+\d{{4}}|\d{{1,2}}/\d{{2,4}}|\b(?:19[5-9]\d|20[0-2]\d)\b)"
    rf"\s*(?:to|until|-|–)\s*"
    rf"({_MONTHS}\s+\d{{4}}|\d{{1,2}}/\d{{2,4}}|\b(?:19[5-9]\d|20[0-2]\d)\b|current|present|now)",
    re.IGNORECASE,
)

_CERT_REGEX = re.compile(
    r"(?:"
    r"(?:certified\s+[\w\s]{2,40}(?:professional|specialist|analyst|manager|associate|consultant|practitioner|engineer|developer|administrator))"
    r"|(?:[\w\s]{1,30}certification)"
    r"|(?:certificate\s+(?:in\s+)?[\w\s]{2,40})"
    r"|(?:license[d]?\s+(?:in\s+)?[\w\s]{2,30})"
    r"|\b(?:cpa|cfa|cma|acca|phr|sphr|shrm-cp|shrm-scp|pmp|cissp|cisa|cism|aws|gcp|azure|six\s+sigma|lean\s+six\s+sigma)\b"
    r")",
    re.IGNORECASE,
)

_SECTION_PATTERNS = {
    "summary": (
        r"(?:summary|professional\s+summary|career\s+summary"
        r"|profile|professional\s+profile|career\s+profile"
        r"|executive\s+profile|objective|career\s+objective|professional\s+objective"
        r"|about\s+me|introduction"
        r"|ringkasan|profil|tentang\s+saya|objektif)"
    ),
    "highlights": (
        r"(?:highlights?|skill\s+highlights?|core\s+competencies"
        r"|accomplishments?|qualifications?|core\s+qualifications?"
        r"|pencapaian|prestasi)"
    ),
    "experience": (
        r"(?:experience|work\s+experience|professional\s+experience"
        r"|work\s+history|employment|employment\s+history"
        r"|internship|internships?|projects?|project\s+experience"
        r"|pengalaman|pengalaman\s+kerja|riwayat\s+pekerjaan)"
    ),
    "education": (
        r"(?:education|educational\s+background|academic\s+background"
        r"|academic\s+qualifications?|qualifications?|academic|degree"
        r"|university|college|training"
        r"|pendidikan|riwayat\s+pendidikan|latar\s+belakang\s+pendidikan)"
    ),
    "certifications": (
        r"(?:certifications?|licenses?\s+and\s+certifications?|certifications?\s+and\s+licenses?"
        r"|professional\s+certifications?|credentials?|certificates?"
        r"|professional\s+development|courses?|training"
        r"|sertifikasi|sertifikat|lisensi|pelatihan)"
    ),
    "skills": (
        r"(?:skills?|technical\s+skills?|core\s+skills?|competencies"
        r"|expertise|proficiencies|technologies|tools?"
        r"|areas\s+of\s+expertise"
        r"|keahlian|keterampilan|kompetensi|kemampuan)"
    ),
}

_SECTION_REGEX = re.compile(
    r"(?:^|(?:\n\s*)|\s{3,})"
    r"("
    + "|".join(v for v in _SECTION_PATTERNS.values() if v)
    + r")"
    r"(?:\s{2,}|\s*[\n\t:])",
    re.IGNORECASE | re.MULTILINE,
)


In [24]:
def _basic_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\xa0", " ").replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\ufeff", "").replace("－", " ")
    text = re.sub(r"\bcompany\s+name\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bcity\s*,?\s*state\b",  "", text, flags=re.IGNORECASE)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _clean_section_text(text: str) -> str:
    text = _basic_clean(text)
    text = text.lower()
    text = re.sub(r"[\u2022\u25cf\u25cb\u25aa\uf0b7\xb7\*\•·]", ",", text)
    text = re.sub(r"[,\s]{2,}", " ", text)
    return text.strip().strip(",").strip()


def _map_to_section(matched_text: str) -> str:
    for key, pattern in _SECTION_PATTERNS.items():
        if pattern and re.fullmatch(pattern, matched_text, re.IGNORECASE):
            return key
    return None


def _extract_sections(text: str) -> dict:
    result = {k: "" for k in _SECTION_PATTERNS}
    result["other"] = ""

    splits = []
    for m in _SECTION_REGEX.finditer(text):
        matched = m.group(1).lower().strip()
        section_key = _map_to_section(matched)
        splits.append((m.start(), m.end(), section_key))

    if not splits:
        result["summary"] = _clean_section_text(text)
        return result

    for i, (start, end, key) in enumerate(splits):
        next_start = splits[i + 1][0] if i + 1 < len(splits) else len(text)
        section_text = text[end:next_start].strip()

        if key == "skills":
            cleaned = _basic_clean(section_text).lower()
        elif key == "certifications":
            cleaned = section_text
        else:
            cleaned = _clean_section_text(section_text)

        if key:
            result[key] = (result[key] + " " + cleaned).strip() if result[key] else cleaned
        else:
            result["other"] = (result["other"] + " " + cleaned).strip()

    return result


def _parse_skills_list(skills_text: str) -> list:
    if not skills_text:
        return []
    items = re.split(r"[,;]", skills_text)
    return [s.strip().lower() for s in items if s.strip() and len(s.strip()) > 1]


def _parse_date(s: str):
    s = s.lower().strip()
    if any(x in s for x in ["current", "present", "now"]):
        now = datetime.now()
        return now.year + now.month / 12
    if "/" in s:
        parts = s.split("/")
        m, y = parts[0], parts[1]
        return int(y) + (int(m) / 12 if m.isdigit() else 0)
    m_y = re.search(rf"({_MONTHS})\s+(\d{{4}})", s)
    if m_y:
        m_map = {m: i + 1 for i, m in enumerate(
            ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"])}
        return int(m_y.group(2)) + m_map[m_y.group(1)[:3]] / 12
    if re.fullmatch(r"(?:19[5-9]\d|20[0-2]\d)", s):
        return float(s)
    return None


def _calculate_total_experience(experience_text: str) -> float:
    if not isinstance(experience_text, str) or not experience_text.strip():
        return 0.0
    seen, total = set(), 0.0
    for m in _DATE_RANGE_REGEX.finditer(experience_text):
        key = (m.group(1).lower().strip(), m.group(2).lower().strip())
        if key in seen:
            continue
        seen.add(key)
        start = _parse_date(m.group(1))
        end   = _parse_date(m.group(2))
        if start and end:
            diff = abs(end - start)
            if diff <= 50:
                total += diff
    return round(total, 1) if total else 0.0


def _extract_certifications(full_text: str, cert_section: str = "") -> list:
    found, seen = [], set()
    if cert_section:
        raw_items = re.split(r"[\n\r]|(?:·|\u2022|\u25cf|\u25aa|\uf0b7|\*)", cert_section)
        if len(raw_items) == 1:
            raw_items = re.split(
                r"(?<=[.)])\s{2,}(?=[A-Z])|(?<=\))\s+(?=[A-Z][a-z]+\s+[A-Z])",
                cert_section,
            )
        for item in raw_items:
            item = _basic_clean(item).lower().strip().strip(",").strip()
            item = re.sub(r"\s+", " ", item)
            if len(item) > 3 and item not in seen:
                seen.add(item)
                found.append(item)
        return found
    for m in _CERT_REGEX.finditer(full_text):
        cert = re.sub(r"\s+", " ", m.group()).strip().lower()
        if len(cert) < 3 or cert in seen:
            continue
        seen.add(cert)
        found.append(cert)
    return found


def _clean_text_for_model(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s,.\'\-+#/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [27]:
!pip install pypdf

In [40]:
def extract_cv_from_pdf(pdf_path: str) -> dict:
    reader   = PdfReader(pdf_path)
    raw_text = "\n".join((page.extract_text() or "") for page in reader.pages).strip()

    if not raw_text:
        raise ValueError(f"Tidak ada teks yang bisa diekstrak dari '{pdf_path}'.")

    sections = _extract_sections(raw_text)

    skills_raw        = sections.get("skills", "") or ""
    skills_list       = _parse_skills_list(skills_raw)
    experience_years  = _calculate_total_experience(sections.get("experience", ""))
    certifications    = _extract_certifications(raw_text, sections.get("certifications", ""))
    has_education     = 1 if sections.get("education", "").strip() else 0

    parts = []

    summary = sections.get("summary", "")
    if summary.strip():
        parts.append(_clean_text_for_model(summary))

    experience = sections.get("experience", "")
    if experience.strip():
        parts.append(_clean_text_for_model(experience[:800]))

    highlights = sections.get("highlights", "")
    if highlights.strip():
        parts.append(_clean_text_for_model(highlights[:300]))

    education = sections.get("education", "")
    if education.strip():
        parts.append(_clean_text_for_model(education[:300]))

    combined_text = " ".join(parts).strip()

    return {
        "text":             combined_text,
        "skills_raw":       skills_raw,
        "skills_list":      skills_list,
        "skills_count":     len(skills_list),
        "experience_years": experience_years,
        "cert_count":       len(certifications),
        "certifications":   certifications,
        "has_education":    has_education,
        "_sections":        sections,
        "_raw_text_len":    len(raw_text),
    }

In [41]:
PDF_PATH ="sample_cv.pdf"

print("=" * 60)
print("  CV PDF INFERENCE")
print("=" * 60)

cv = extract_cv_from_pdf(PDF_PATH)

print(f"\n📄 File          : {PDF_PATH}")
print(f"   Teks diekstrak: {cv['_raw_text_len']:,} karakter")
print(f"   Pengalaman    : {cv['experience_years']} tahun")
print(f"   Sertifikasi   : {cv['cert_count']} item  → {cv['certifications']}")
print(f"   Ada pendidikan: {'Ya' if cv['has_education'] else 'Tidak'}")
print(f"   Skills ({cv['skills_count']}): {cv['skills_raw'][:120]}{'...' if len(cv['skills_raw']) > 120 else ''}")

top3 = predict_top3_cv(
    text        = cv["text"],
    exp_years   = cv["experience_years"],
    certs_count = cv["cert_count"],
    has_edu     = cv["has_education"],
    skills_raw  = cv["skills_raw"],
)

print("\n" + "=" * 60)
print("  TOP-3 ROLE MATCHES")
print("=" * 60)
for r in top3:
    conf  = float(r["final_score"])
    bar   = "█" * int(conf / 5)
    print(f"  #{r['rank']}  {r['role']:<35s}  {conf:6.2f}%  {bar}")
print("=" * 60)

  CV PDF INFERENCE

📄 File          : sample_cv.pdf
   Teks diekstrak: 5,822 karakter
   Pengalaman    : 17.4 tahun
   Sertifikasi   : 0 item  → []
   Ada pendidikan: Ya
   Skills (45): account reconciliation, accounting, general accounting, accruals, ad, agency, balance sheets, billing, budgets, budget, ...

  TOP-3 ROLE MATCHES
  #1  Accounting & Tax                      53.49%  ██████████
  #2  Corporate Finance & Banking           36.58%  ███████
  #3  Supply Chain & Operations              4.20%  


# Skill Gap Database & Analysis

In [42]:
def analyze_skill_gap(cv_skills_raw: str, full_cv_text: str, predicted_role: str) -> dict:
    cv_skills = {s.strip().lower() for s in cv_skills_raw.split(',') if s.strip()}
    required  = skill_db.get(predicted_role, [])
    full_text_lower = f" {full_cv_text.lower()} " 

    matched = []
    missing = []
    
    for req in required:
        req_lower = req.lower()
        if req_lower in cv_skills:
            matched.append(req)
        elif f" {req_lower} " in full_text_lower or f" {req_lower}," in full_text_lower:
            matched.append(req)
        else:
            missing.append(req)

    coverage = round(len(matched) / len(required) * 100, 1) if required else 0

    return {
        'predicted_role':     predicted_role,
        'required_skills':    required,
        'matched_skills':     matched,
        'missing_skills':     missing[:10],
        'coverage_pct':       coverage,
        'progress_bar_value': round(coverage / 100, 3)
    }

print("\n=== SKILL GAP ANALYSIS ===")

for r in top3:
    gap = analyze_skill_gap(cv["skills_raw"], cv["text"], r['role'])
    
    print(f"\n#{r['rank']} {r['role']} ({r['final_score']:.1f}%)")
    print(f"   ✅ Matched  : {gap.get('matched_skills', [])}")
    print(f"   ❌ Missing  : {gap.get('missing_skills', [])}")
    print(f"   📊 Coverage : {gap.get('coverage_pct', 0)}%")


=== SKILL GAP ANALYSIS ===

#1 Accounting & Tax (53.5%)
   ✅ Matched  : ['accounting', 'financial reporting', 'financial statements', 'microsoft office', 'general ledger', 'account reconciliation', 'financial', 'outlook', 'reporting', 'processes']
   ❌ Missing  : ['quickbooks', 'excel', 'accounts payable', 'word', 'payroll', 'powerpoint', 'sales', 'sap', 'accounts receivable', 'finance']
   📊 Coverage : 50.0%

#2 Corporate Finance & Banking (36.6%)
   ✅ Matched  : ['financial', 'customer service', 'accounting', 'budget', 'reporting', 'documentation', 'policies', 'financial analysis']
   ❌ Missing  : ['excel', 'sales', 'leadership', 'clients', 'credit', 'finance', 'powerpoint', 'banking', 'time management', 'client']
   📊 Coverage : 40.0%

#3 Supply Chain & Operations (4.2%)
   ✅ Matched  : ['customer service', 'microsoft office']
   ❌ Missing  : ['project management', 'team leadership', 'policies and procedures implementation', 'solutions development', 'staff training and development'

# AI Career Coach (Mistral API)

In [43]:
from dotenv import load_dotenv
load_dotenv() 

def get_ai_career_coach(top1_role: str, top1_score: float, exp_years: float,
                        skills_raw: str, missing_skills: list) -> str:
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "")
    if not MISTRAL_API_KEY:
        return "⚠️  Belum set API key. Pastikan file .env sudah dibuat dengan benar."

    prompt = f"""Anda adalah KAVA (Karier AI Validasi Asisten), asisten karir profesional.

Profil Kandidat:
- Role diprediksi (Top-1): {top1_role} ({top1_score}% final score)
- Pengalaman: {exp_years} tahun
- Skills yang dimiliki: {skills_raw}
- Skills yang perlu ditingkatkan: {', '.join(missing_skills)}

Berikan dalam Bahasa Indonesia:
1. **Saran Karir**: Langkah konkret berikutnya untuk kandidat ini
2. **Skill yang Harus Dipelajari**: Dari daftar missing skills, pilih 3 yang paling penting dan jelaskan mengapa
3. **Rekomendasi Sertifikasi**: 1 sertifikasi internasional yang paling relevan untuk menembus pasar kerja {top1_role}

Format respons: ringkas, actionable, maksimal 200 kata."""
    
    try:
        response = requests.post(
            "https://api.mistral.ai/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {MISTRAL_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "mistral-small-latest",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 400
            },
            timeout=30
        )
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

    except requests.exceptions.HTTPError as e:
        return f"❌ HTTP Error {response.status_code}: {response.text}"
    except Exception as e:
        return f"❌ Error: {str(e)}"

if top3:
    gap_top1 = analyze_skill_gap(cv["skills_raw"], cv["text"], top3[0]['role'])
    
    coach = get_ai_career_coach(
        top1_role      = top3[0]['role'],
        top1_score     = top3[0]['final_score'],
        exp_years      = cv['experience_years'],
        skills_raw     = cv['skills_raw'] if cv['skills_raw'] else "Tidak dituliskan spesifik",
        missing_skills = gap_top1.get('missing_skills', [])
    )
    print("=== AI CAREER COACH (Mistral) ===")
    print(coach)

=== AI CAREER COACH (Mistral) ===
**1. Saran Karir:**
- **Spesialisasi:** Fokus pada peran *Senior Accountant* atau *Tax Specialist* di perusahaan besar/korporat untuk memanfaatkan pengalaman 17 tahun.
- **Industri:** Targetkan sektor *manufaktur, keuangan, atau konsultan pajak* yang membutuhkan keahlian *compliance reporting* dan *financial analysis*.
- **Pindah Perusahaan:** Pertimbangkan perusahaan dengan sistem ERP (Oracle, SAP) untuk meningkatkan *financial management* skills.
- **Networking:** Bergabung dengan asosiasi akuntansi (e.g., IAI) atau LinkedIn groups untuk peluang *referral*.

**2. Skill yang Harus Dipelajari (Prioritas):**
1. **QuickBooks** – Standar *accounting software* untuk UKM dan startup, meningkatkan daya saing di pasar.
2. **Excel (Advanced)** – Analisis data, *pivot table*, dan *VBA* krusial untuk *financial reporting* dan pengambilan keputusan.
3. **Accounts Payable/Receivable** – Keterampilan dasar akuntansi operasional yang sering dicari di low/mid-level r

# Simpan Semua Artifacts

In [44]:
SAVE_DIR = "./artifacts/"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Menyimpan ke: {SAVE_DIR}")

vec_payload = {'config': vectorizer.get_config(), 'weights': vectorizer.get_weights()}
with open(f"{SAVE_DIR}vectorizer.pkl", "wb") as f:
    pickle.dump(vec_payload, f)

with open(f"{SAVE_DIR}scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(f"{SAVE_DIR}label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

with open(f"{SAVE_DIR}skill_gap_db.json", "w") as f:
    json.dump(skill_db, f, indent=2)
    
with open(f"{SAVE_DIR}tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vec, f)
    
with open(f"{SAVE_DIR}domain_vectors.pkl", "wb") as f:
    pickle.dump(domain_vectors, f)

metadata = {
    'model_name':           'KAVA v3.0',
    'num_classes':          num_classes,
    'class_names':          le.classes_.tolist(),
    'max_len':              MAX_LEN,
    'embed_dim':            EMBED_DIM,
    'max_tokens':           MAX_TOKENS,
    'num_numeric_features': NUM_NUMERIC_FEATURES,
    'numeric_columns':      NUMERIC_COLS,
    'architecture':         'BiLSTM(64) + AttentionPooling(64) + Numeric Fusion',
    'test_accuracy':        round(float(acc), 4),
    'macro_f1':             round(float(macro_f1), 4),
    'mae':                  round(float(final_mae), 5),
    'trained_at':           datetime.now().isoformat()
}
with open(f"{SAVE_DIR}model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Training history
with open(f"{SAVE_DIR}training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\n✅ Artifact yang dihasilkan:")
artifacts = [
    'kava_best_model.keras', 'vectorizer.pkl', 'scaler.pkl',
    'label_encoder.pkl', 'skill_gap_db.json', 
    'model_metadata.json', 'training_history.json', 
    'tfidf_vectorizer.pkl', 'domain_vectors.pkl'
]
for fname in artifacts:
    fpath = f"{SAVE_DIR}{fname}"
    size  = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
    print(f"   {fname:40s} {size:8.1f} KB")

Menyimpan ke: ./artifacts/

✅ Artifact yang dihasilkan:
   kava_best_model.keras                     15936.4 KB
   vectorizer.pkl                              282.4 KB
   scaler.pkl                                    0.6 KB
   label_encoder.pkl                             0.9 KB
   skill_gap_db.json                             9.1 KB
   model_metadata.json                           1.3 KB
   training_history.json                         2.5 KB
   tfidf_vectorizer.pkl                         18.2 KB
   domain_vectors.pkl                           16.6 KB


# End-to-End Test + Format Response untuk Fullstack

In [45]:
sample_payload = {
    "text":             cv["text"],
    "skills_raw":       cv["skills_raw"],
    "experience_years": cv["experience_years"],
    "cert_count":       cv["cert_count"],
    "has_education":    cv["has_education"],
    "has_highlights":   1  # Asumsi default untuk testing
}

top3_result = predict_top3_cv(
    text        = sample_payload["text"],
    exp_years   = sample_payload["experience_years"],
    certs_count = sample_payload["cert_count"],
    has_edu     = sample_payload["has_education"],
    skills_raw  = sample_payload["skills_raw"]
)

gap_result = analyze_skill_gap(
    cv_skills_raw  = sample_payload["skills_raw"], 
    full_cv_text   = sample_payload["text"], 
    predicted_role = top3_result[0]["role"]
)

full_response = {
    "status":             "success",
    "top3_role_matches":  top3_result,
    "skill_gap":          gap_result,
    "ai_career_coach":    get_ai_career_coach(
        top1_role      = top3_result[0]["role"],
        top1_score     = top3_result[0]["final_score"],
        exp_years      = sample_payload["experience_years"],
        skills_raw     = sample_payload["skills_raw"] if sample_payload["skills_raw"] else "Tidak dituliskan spesifik",
        missing_skills = gap_result.get("missing_skills", [])
    )
}

print("=== RESPONSE JSON UNTUK TIM FULLSTACK ===")
print(json.dumps(full_response, indent=2, ensure_ascii=False))

=== RESPONSE JSON UNTUK TIM FULLSTACK ===
{
  "status": "success",
  "top3_role_matches": [
    {
      "rank": 1,
      "role": "Accounting & Tax",
      "dl_confidence": 56.99,
      "keyword_score": 39.5,
      "final_score": 53.49
    },
    {
      "rank": 2,
      "role": "Corporate Finance & Banking",
      "dl_confidence": 39.42,
      "keyword_score": 25.2,
      "final_score": 36.58
    },
    {
      "rank": 3,
      "role": "Supply Chain & Operations",
      "dl_confidence": 0.1,
      "keyword_score": 20.6,
      "final_score": 4.2
    }
  ],
  "skill_gap": {
    "predicted_role": "Accounting & Tax",
    "required_skills": [
      "quickbooks",
      "excel",
      "accounts payable",
      "accounting",
      "financial reporting",
      "financial statements",
      "microsoft office",
      "general ledger",
      "word",
      "account reconciliation",
      "financial",
      "payroll",
      "powerpoint",
      "outlook",
      "reporting",
      "sales",
      "sap"